# analysis — baseline / sweep plots

Reads only `metrics.jsonl` from each branch's run directory (Drive-mounted, or local `runs/` if you copied them down). Unlike `run_colab.ipynb`, this notebook is allowed to be exploratory — plotting is inherently interactive, this is the "looking" notebook, not the "launching" one.

Scope note: only `baseline` and `sweep_a8` exist right now — the PI-controller / heuristic comparisons (Figure 4, P-dominance, grad-norm windows) come back once those branches are implemented.

In [ ]:
import matplotlib.pyplot as plt
from pidlora.logging_utils import read_metrics_jsonl

RUNS_DIR = "runs"  # point this at the Drive mount if running on Colab, e.g.
                    # "/content/drive/MyDrive/llm-pid-alignment/runs"

BRANCH_METRICS = {
    "baseline": f"{RUNS_DIR}/baseline_alpha16/metrics.jsonl",
    "sweep_a8": f"{RUNS_DIR}/sweep_alpha8/metrics.jsonl",
}

records = {}
for name, path in BRANCH_METRICS.items():
    try:
        records[name] = read_metrics_jsonl(path)
    except FileNotFoundError:
        print(f"skipping {name!r}: {path} not found yet")

def events(branch, event):
    return [r for r in records.get(branch, []) if r["event"] == event]

## Figure 1 — KL divergence over training steps (control set, baseline)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
kl = events("baseline", "kl_eval")
if kl:
    ax.plot([r["step"] for r in kl], [r["kl_raw"] for r in kl], label="baseline (raw)", alpha=0.5)
    ax.plot([r["step"] for r in kl], [r["kl_filt"] for r in kl], label="baseline (EMA)")
ax.set_xlabel("step"); ax.set_ylabel("KL"); ax.set_title("Figure 1 — KL vs step")
ax.legend(); plt.tight_layout(); plt.show()

## Figure 2 — Perplexity over training steps (held-out set, baseline)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ho = events("baseline", "holdout_eval")
if ho:
    ax.plot([r["step"] for r in ho], [r["perplexity"] for r in ho], marker="o", label="baseline")
ax.set_xlabel("step"); ax.set_ylabel("perplexity"); ax.set_title("Figure 2 — Held-out perplexity vs step")
ax.legend(); plt.tight_layout(); plt.show()

## Figure 3 — Training loss, all branches

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for branch in records:
    tr = events(branch, "train_step")
    if tr:
        ax.plot([r["step"] for r in tr], [r["loss"] for r in tr], label=branch, alpha=0.8)
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title("Figure 3 — Training loss")
ax.legend(); plt.tight_layout(); plt.show()

## Figure 5 — Pareto front: mean loss vs mean KL (last 50 steps, EMA)

In [ ]:
def tail_mean_by_step(records_with_step, value_key, window_steps=50):
    """Averages over the last `window_steps` of *training steps*, not the last N list
    entries. Loss is logged every step, so N=50 entries happens to equal 50 steps — but
    KL is logged every kl_eval_every steps (25 by default), so 50 *entries* of KL would
    span 50*25=1250 steps, silently averaging over the entire 1000-step run instead of
    the intended tail window. Filtering by step avoids that logging-cadence dependence.
    """
    if not records_with_step:
        return float("nan")
    max_step = max(r["step"] for r in records_with_step)
    window = [r[value_key] for r in records_with_step if r["step"] >= max_step - window_steps]
    return sum(window) / max(len(window), 1)

pareto_points = {}
for branch in ["baseline", "sweep_a8"]:
    loss_records = events(branch, "train_step")
    kl_records = events(branch, "kl_eval") or events(branch, "final_eval")
    if loss_records and kl_records:
        pareto_points[branch] = (
            tail_mean_by_step(loss_records, "loss"),
            tail_mean_by_step(kl_records, "kl_raw"),
        )

fig, ax = plt.subplots(figsize=(6, 5))
for branch, (loss, kl) in pareto_points.items():
    ax.scatter(kl, loss, s=80)
    ax.annotate(branch, (kl, loss), textcoords="offset points", xytext=(6, 6))
ax.set_xlabel("mean KL (last 50 steps)"); ax.set_ylabel("mean loss (last 50 steps)")
ax.set_title("Figure 5 — Pareto front (baseline vs sweep_a8; PI/heuristic points come later)")
plt.tight_layout(); plt.show()
pareto_points

## Figure 6 — Preference margin over training steps (baseline)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ho = events("baseline", "holdout_eval")
if ho:
    ax.plot([r["step"] for r in ho], [r["preference_margin"] for r in ho], marker="o", label="baseline")
ax.axhline(0.0, color="black", linestyle=":")
ax.set_xlabel("step"); ax.set_ylabel("mean logP(chosen) - logP(rejected)")
ax.set_title("Figure 6 — Preference margin vs step")
ax.legend(); plt.tight_layout(); plt.show()